# Setting up training environment

In [ ]:
#!pip install pquant-ml[torch]
!pip install hls4ml
!pip install torchvision torchinfo

In [ ]:
import os
# Do this before importing Keras to use Torch PQLayers
os.environ["KERAS_BACKEND"] = "torch"
import random
import numpy as np
import torch.nn.functional as F
import numpy as np
from torch.utils.data import TensorDataset, DataLoader
import pquant
import torch

import keras
# Pruning layers implemented using channels first format
keras.backend.set_image_data_format("channels_first")
from pquant import get_ebops
import random

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
set_seed(42)

## Fetch the Jet tagging dataset from Open ML


In [ ]:
import pickle as pkl
from pathlib import Path

from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
np.random.seed(42)
random.seed(42)

def get_data(data_path: Path, seed=42):
    try:
        import zstd
    except ImportError:
        zstd = None
    if not os.path.exists(data_path):
        print('Downloading data...')
        data = fetch_openml('hls4ml_lhc_jets_hlf')
        buf = pkl.dumps(data)
        os.makedirs(data_path.parent, exist_ok=True)
        with open(data_path, 'wb') as f:
            if zstd is not None:
                buf = zstd.compress(buf)
            f.write(buf)
    else:
        with open(data_path, 'rb') as f:
            buf = f.read()
            if zstd is not None:
                buf = zstd.decompress(buf)
            data = pkl.loads(buf)

    X, y = data['data'], data['target']
    codecs = {'g': 0, 'q': 1, 't': 4, 'w': 2, 'z': 3}
    y = np.array([codecs[i] for i in y])

    X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=seed)
    scaler = StandardScaler()
    X_train_val = scaler.fit_transform(X_train_val).astype(np.float32)
    X_test = scaler.transform(X_test).astype(np.float32)

    N_train = int(0.9 * len(X_train_val))
    X_train, X_val = X_train_val[:N_train], X_train_val[N_train:]
    y_train, y_val = y_train_val[:N_train], y_train_val[N_train:]

    return (X_train, y_train), (X_val, y_val), (X_test, y_test)

(X_train, y_train), (X_val, y_val), (X_test, y_test) = get_data(Path('/tmp/inp_data.zst'))
print(f"train: {len(X_train)}  validation: {len(X_val)}  test: {len(X_test)}")

In [ ]:
import torch
import torch.nn.functional as F

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


class GPUBatches:

    def __init__(self, X, y, batch_size, shuffle, device=DEVICE, seed=42):
        self.X, self.y = X.to(device), y.to(device)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.generator = torch.Generator().manual_seed(seed)

    def __len__(self):
        return (len(self.X) + self.batch_size - 1) // self.batch_size

    def __iter__(self):
        n = len(self.X)
        if self.shuffle:
            order = torch.randperm(n, generator=self.generator).to(self.X.device)
        else:
            order = torch.arange(n, device=self.X.device)
        for start in range(0, n, self.batch_size):
            idx = order[start : start + self.batch_size]
            yield self.X[idx], self.y[idx]


def create_dataloaders(batch_size):
    def to_tensors(X, y):
        return torch.from_numpy(X).float(), F.one_hot(torch.from_numpy(y).long(), num_classes=5).float()

    train_loader = GPUBatches(*to_tensors(X_train, y_train), batch_size, shuffle=True)
    val_loader = GPUBatches(*to_tensors(X_val, y_val), batch_size, shuffle=False)
    test_loader = GPUBatches(*to_tensors(X_test, y_test), batch_size, shuffle=False)
    return train_loader, val_loader, test_loader

## Define the PQuantML config and construct the model
The architecture is a dense network with 3 hidden layers with 64,32,32 units, followed by an output layer of 5 units. In PyTorch, no PQActivation class is needed for ReLU.

In [ ]:
from pquant import dst_config
from pquant.layers import PQDense
from pquant.activations import PQActivation

def build_model(config, prune_output=True, in_out_quant_bits=(1,4,11)):
  # Training data range between -6.18 and 8.96, set 4 integer bits for model input quantizer above to cover whole range
    class Model(torch.nn.Module):
        def __init__(self, config, prune_output=True, in_out_quant_bits = (1,4,11)):
            super().__init__()
            self.dense1 = PQDense(config, 16, 64,
                                  in_quant_bits = in_out_quant_bits)
            self.dense2 = PQDense(config, 64, 32)
            self.dense3 = PQDense(config, 32, 32)
            self.dense4 = PQDense(config, 32, 5,
                                  quantize_output=True,
                                  enable_pruning=prune_output,
                                  out_quant_bits=in_out_quant_bits)
            self.relu1 = PQActivation(config, "relu", quantize_input=False)
            self.relu2 = PQActivation(config, "relu", quantize_input=False)
            self.relu3 = PQActivation(config, "relu", quantize_input=False)


        def forward(self, x):
            x = self.relu1(self.dense1(x))
            x = self.relu2(self.dense2(x))
            x = self.relu3(self.dense3(x))
            x = self.dense4(x)
            return x
    return Model(config, prune_output, in_out_quant_bits)



## Create the training and validation loops
Create simple training and validation loops that save some statistics during validation to an array for plotting purposes

In [ ]:
from pquant import get_layer_keep_ratio, get_model_losses

# Per-epoch statistics gathered by the loops below for plotting. They are module-level lists, so call
# reset_history() before each training run; otherwise the plots show every previous run as well.
train_accuracies, ebops, val_accuracies, val_losses, remaining_weights = [], [], [], [], []

def reset_history():
    for history in (train_accuracies, ebops, val_accuracies, val_losses, remaining_weights):
        history.clear()


def training_loop(model, trainloader, device, loss_function, optimizer, epoch, *args, **kwargs):
    accuracies = []
    for data in trainloader:
        inputs, labels = data
        inputs, labels = inputs.to(device), labels.to(device)
        inputs = torch.reshape(inputs, (inputs.shape[0], -1))
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = loss_function(outputs, labels)
        losses = get_model_losses(model, torch.tensor(0.).to(device)) # Add pruning layer losses, HGQ loss if HGQ enabled
        loss += losses
        loss.backward()
        optimizer.step()
        acc = torch.mean((torch.argmax(outputs, dim=1) == torch.argmax(labels, dim=1)).float()).cpu().numpy()
        accuracies.append(acc)
    accuracy = np.mean(accuracies)
    train_accuracies.append(accuracy)

def validate_loop(model, valloader, device, loss_function, epoch, gather_ebops, *args, **kwargs):
    val_loss = []
    accuracies = []
    with torch.no_grad():
        for data in valloader:
            inputs, labels = data
            inputs, labels = inputs.to(device), labels.to(device)
            inputs = torch.reshape(inputs, (inputs.shape[0], -1))
            outputs = model(inputs)
            loss = loss_function(outputs, labels)
            losses = get_model_losses(model, torch.tensor(0.).to(device))
            loss += losses
            val_loss.append(loss.item())
            acc = torch.mean((torch.argmax(outputs, dim=1) == torch.argmax(labels, dim=1)).float()).cpu().numpy()
            accuracies.append(acc)
        # Gather statistics per epoch
        if gather_ebops:
          ebops.append(get_ebops(model).cpu().numpy())
        ratio = get_layer_keep_ratio(model)
        remaining_weights.append(ratio.cpu().numpy())
        accuracy = np.mean(accuracies)
        val_accuracies.append(accuracy)
        loss_val = np.mean(val_loss)
        val_losses.append(loss_val)


# DST with fixed point quantization

## Create the config

In [ ]:
from pquant import dst_config
# Training data range between -6.18 and 8.96, set 4 integer bits for model input quantizer above to cover whole range
config = dst_config()
config.training_parameters.epochs = 100
config.training_parameters.fine_tuning_epochs = 50
# 8 bit data and weights
# For data default k=0, weights k=1
config.quantization_parameters.default_data_integer_bits = 3.
config.quantization_parameters.default_data_fractional_bits = 5.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 7.
config.quantization_parameters.overflow_mode_data = "SAT"
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.dynamic_data_quantization = True
config.quantization_parameters.granularity = "per_weight"
#config.pruning_parameters.enable_pruning = False
# DST hyperparameter, higher value means higher sparsity
config.pruning_parameters.alpha = 1e-4

## Build the model and put to GPU

In [ ]:
model = build_model(config)
model(torch.rand(1, 16)) # Call once to build Keras layers
model.to("cuda") # Created layers might not be in CUDA, do this again just in case
# Every training stage and train/eval mode gets its own compiled graph; raise the limit so dynamo does not fall back to eager
torch._dynamo.config.cache_size_limit = 64
# dynamic=True: the smaller last batch of an epoch does not trigger a recompile
compiled_model = torch.compile(model, dynamic=True)


## Train the model

In [ ]:
from pquant import train_model
reset_history()  # start the plots from this run
BATCH_SIZE = 1024 # High for tutorial purposes, should be e.g. 1024
train_loader, val_loader, test_loader = create_dataloaders(BATCH_SIZE)
loss_function = torch.nn.CrossEntropyLoss()
#optimizer = torch.optim.Adam(lr=1e-3, params=model.parameters(), weight_decay=1e-4)

# This can work better with DST and CS and AutoSparse. Exclude learned threshold from weight_decay
parameters = list(compiled_model.named_parameters())
threshold_params = [v for n, v in parameters if "threshold" in n and v.requires_grad]
rest_params = [v for n, v in parameters if "threshold" not in n and v.requires_grad]
optimizer = torch.optim.Adam([
                {
                    "params": threshold_params,
                    "weight_decay": 0,
                },
                {"params": rest_params, "weight_decay": 1e-4},
            ],
            lr=1e-3)


from datetime import datetime
start = datetime.now()
trained_model = train_model(model = compiled_model,
                            config = config,
                            train_func = training_loop,
                            valid_func = validate_loop,
                            trainloader = train_loader,
                            device="cuda",
                            valloader = val_loader,
                            loss_function = loss_function,
                            optimizer = optimizer,
                            input_shape=(16,),
                            gather_ebops=True
                                )
end = datetime.now()
print("Finished training, took", end-start)

## Apply compression to weights one more time
The weights and biases during training are in their non-compressed state, only going through compression during the forward pass. Run the function below to compress the weights and biases one more time, and override the previous non-compressed state of weights and biases.

In [ ]:
from pquant import apply_final_compression
apply_final_compression(trained_model)

## Convert the model to FPGA firmware with hls4ml
The hls4ml converter extracts the quantization parameters from the PQLayers during the conversion.

In [ ]:
from hls4ml.utils import config_from_pytorch_model
from hls4ml.converters import convert_from_pytorch_model
model = trained_model._orig_mod # Run hls4ml conversion with the original eager model instead of the compiled model
model.eval()
output_dir = f"example_result_dst"
os.makedirs(output_dir, exist_ok = True)

io_type = 'io_parallel'
backend = 'vitis'
target = 'xcvu13p-flga2577-2-e'
model.to("cpu")
hls_config = config_from_pytorch_model(
                model,
                input_shape=tuple([16]),
                granularity='name',
                backend=backend,
                transpose_outputs=True
            )
hls_model = convert_from_pytorch_model(
                model,
                io_type=io_type,
                output_dir=output_dir,
                backend=backend,
                hls_config=hls_config,
                part=target,
            )
hls_model.compile()
res_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml_config = np.mean((np.argmax(res_hls, axis=1) == y_test))

model.to("cuda")
p_torch = model(torch.tensor(X_test).to("cuda")).detach().cpu().numpy()
p_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml = np.mean((np.argmax(p_hls, axis=1) == y_test))
test_accuracy_torch = np.mean((np.argmax(p_torch, axis=1) == y_test))

print(f"Accuracies: hls4ml={test_accuracy_hls4ml_config}, keras={test_accuracy_torch}")


## Make some plots from values gathered during training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(2,2, figsize=(10,10))

ax[0,0].plot(remaining_weights, label="Remaining weights %")
ax[0,0].legend()
ax[0,1].plot(val_accuracies, label="Val accuracy")
ax[0,1].legend()
ax[1,0].plot(val_losses, label="Val loss")
ax[1,0].legend()
ax[1,1].plot(ebops, label="EBOPs")
ax[1,1].legend()
plt.show()


# FITCompress

## Create the config

In [ ]:
from pquant import fitcompress_config
config = fitcompress_config()
# How much to train before FITCompress
config.training_parameters.pretraining_epochs = 20
# How much to train post FITCompress
config.training_parameters.epochs = 50

# 8 bit data, weights determined by fitcompress
# For data default k=0
config.quantization_parameters.default_data_integer_bits = 3.
config.quantization_parameters.default_data_fractional_bits = 5.

config.quantization_parameters.overflow_mode_data = "SAT"
config.quantization_parameters.overflow_mode_parameters = "SAT"

# Which bit-widths to consider for weights
config.fitcompress_parameters.quantization_schedule = [4., 5., 6., 7., 8.]
# Which sparsities per layer to consider. Calculated as 1 - np.logspace(start, end, base=10, num=steps)
config.fitcompress_parameters.pruning_schedule.start = 0
config.fitcompress_parameters.pruning_schedule.end = -1
config.fitcompress_parameters.pruning_schedule.steps = 100

# Target % of BOPS remaining after FITCompress
config.fitcompress_parameters.compression_goal = 0.01

# A* by default discards other candidates as soon as next best candidate is found
# Uncomment below line to disable this behavior
#config.fitcompress_parameters.greedy_astar = False

## Build the model and put to GPU

In [ ]:
model = build_model(config)
model(torch.rand(1, 16)) # Call once to build Keras layers
model.to("cuda") # Created Keras layers might not be in CUDA, do this again just in case
torch._dynamo.config.cache_size_limit = 64
compiled_model = torch.compile(model, dynamic=True)

## Train the model

In [ ]:
from pquant import train_model
reset_history()  # start the plots from this run
BATCH_SIZE = 1024
train_loader, val_loader, test_loader = create_dataloaders(BATCH_SIZE)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lr=1e-3, params=compiled_model.parameters())

trained_model = train_model(model = compiled_model,
                            config = config,
                            train_func = training_loop,
                            valid_func = validate_loop,
                            trainloader = train_loader,
                            device="cuda",
                            valloader = val_loader,
                            loss_function = loss_function,
                            optimizer = optimizer,
                            input_shape=(16,),
                            gather_ebops=False
                                )


## Apply compression to weights one more time
The weights and biases during training are in their non-compressed state, only going through compression during the forward pass. Run the function below to compress the weights and biases one more time, and override the previous non-compressed state of weights and biases.

In [ ]:
from pquant import apply_final_compression
apply_final_compression(trained_model)

## Convert the model to FPGA firmware with hls4ml
The hls4ml converter extracts the quantization parameters from the PQLayers during the conversion.

In [ ]:
from hls4ml.utils import config_from_pytorch_model
from hls4ml.converters import convert_from_pytorch_model
model = trained_model._orig_mod # Run hls4ml conversion with the original eager model instead of the compiled model
model.eval()
output_dir = f"example_result_fitcompress"
os.makedirs(output_dir, exist_ok = True)

io_type = 'io_parallel'
backend = 'vitis'
target = 'xcvu13p-flga2577-2-e'
model.to("cpu")
hls_config = config_from_pytorch_model(
                model,
                input_shape=tuple([16]),
                granularity='name',
                backend=backend,
                transpose_outputs=True
            )
hls_model = convert_from_pytorch_model(
                model,
                io_type=io_type,
                output_dir=output_dir,
                backend=backend,
                hls_config=hls_config,
                part=target,
            )
hls_model.compile()
res_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml_config = np.mean((np.argmax(res_hls, axis=1) == y_test))

model.to("cuda")
p_torch = model(torch.tensor(X_test).to("cuda")).detach().cpu().numpy()
p_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml = np.mean((np.argmax(p_hls, axis=1) == y_test))
test_accuracy_torch = np.mean((np.argmax(p_torch, axis=1) == y_test))

print(f"Accuracies: hls4ml={test_accuracy_hls4ml_config}, keras={test_accuracy_torch}")


## Make some plots from values gathered during training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,3, figsize=(20,5))

ax[0].plot(remaining_weights, label="Remaining weights %")
ax[0].legend()
ax[1].plot(val_accuracies, label="Val accuracy")
ax[1].legend()
ax[2].plot(val_losses, label="Val loss")
ax[2].legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

# FITCompress picks one bitwidth per layer (per_tensor quantization) and a pruning mask per layer
weight_bits, input_bits, sparsities = {}, {}, {}
for n, m in model.named_modules():
    if isinstance(m, PQDense):
        k, i, f = (float(np.asarray(t.detach().cpu()).mean()) for t in m.get_weight_quantization_bits())
        weight_bits[n] = k + i + f
        k, i, f = (float(np.asarray(t.detach().cpu()).mean()) for t in m.get_input_quantization_bits())
        input_bits[n] = k + i + f
        weight = m.weight.detach()
        sparsities[n] = float((weight == 0).sum() / weight.numel())

names = list(weight_bits)
x = np.arange(len(names))
fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(x - 0.2, [weight_bits[n] for n in names], width=0.4, label="weights")
ax.bar(x + 0.2, [input_bits[n] for n in names], width=0.4, label="inputs")
ax.set_xticks(x, names)
ax.set_ylabel("bits")
ax.set_title("bitwidth per layer")
ax.legend()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(names, [100 * sparsities[n] for n in names])
ax.set_ylabel("zero weights (%)")
ax.set_title("sparsity per layer")
plt.show()


# Activation Pruning

## Create the config

In [ ]:
from pquant import ap_config
config = ap_config()
config.training_parameters.epochs = 200
config.training_parameters.pretraining_epochs = 20
config.training_parameters.fine_tuning_epochs = 400
# 8 bit data and weights
# For data default k=0
config.quantization_parameters.default_data_integer_bits = 3.
config.quantization_parameters.default_data_fractional_bits = 5.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 7.
config.quantization_parameters.overflow_mode_data = "SAT"
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.granularity = "per_channel"
# How many batches to use to prune
config.pruning_parameters.t_delta = 1
# How often to prune (prune every t_start_collecting_batch epochs)
config.pruning_parameters.t_start_collecting_batch = 5
# If neuron is active less than 80% of the time, prune it
config.pruning_parameters.threshold = 0.8

## Build the model and put to GPU

In [ ]:
model = build_model(config, prune_output=False)
model(torch.rand(1, 16)) # Call once to build Keras layers
model.to("cuda") # Created layers might not be in CUDA, do this again just in case
torch._dynamo.config.cache_size_limit = 64
compiled_model = torch.compile(model, dynamic=True)

## Train the model

In [ ]:
from pquant import train_model
reset_history()  # start the plots from this run
BATCH_SIZE = 8192
train_loader, val_loader, test_loader = create_dataloaders(BATCH_SIZE)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lr=1e-3, params=compiled_model.parameters())

trained_model = train_model(model = compiled_model,
                            config = config,
                            train_func = training_loop,
                            valid_func = validate_loop,
                            trainloader = train_loader,
                            device="cuda",
                            valloader = val_loader,
                            loss_function = loss_function,
                            optimizer = optimizer,
                            input_shape=(16,),
                            gather_ebops=True
                                )


## Apply compression to weights one more time
The weights and biases during training are in their non-compressed state, only going through compression during the forward pass. Run the function below to compress the weights and biases one more time, and override the previous non-compressed state of weights and biases.

In [ ]:
from pquant import apply_final_compression
apply_final_compression(trained_model)

## Convert the model toFPGA firmware with hls4ml
The hls4ml converter extracts the quantization parameters from the PQLayers during the conversion.

In [ ]:
from hls4ml.utils import config_from_pytorch_model
from hls4ml.converters import convert_from_pytorch_model
model = trained_model._orig_mod # Run hls4ml conversion with the original eager model instead of the compiled model
model.eval()
output_dir = f"example_result_ap"
os.makedirs(output_dir, exist_ok = True)

io_type = 'io_parallel'
backend = 'vitis'
target = 'xcvu13p-flga2577-2-e'
model.to("cpu")
hls_config = config_from_pytorch_model(
                model,
                input_shape=tuple([16]),
                granularity='name',
                backend=backend,
                transpose_outputs=True
            )
hls_model = convert_from_pytorch_model(
                model,
                io_type=io_type,
                output_dir=output_dir,
                backend=backend,
                hls_config=hls_config,
                part=target,
            )
hls_model.compile()
res_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml_config = np.mean((np.argmax(res_hls, axis=1) == y_test))

model.to("cuda")
p_torch = model(torch.tensor(X_test).to("cuda")).detach().cpu().numpy()
p_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml = np.mean((np.argmax(p_hls, axis=1) == y_test))
test_accuracy_torch = np.mean((np.argmax(p_torch, axis=1) == y_test))

print(f"Accuracies: hls4ml={test_accuracy_hls4ml_config}, keras={test_accuracy_torch}")


## Make some plots from values gathered during training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,3, figsize=(10,10))

ax[0].plot(remaining_weights, label="Remaining weights %")
ax[0].legend()
ax[1].plot(val_accuracies, label="Val accuracy")
ax[1].legend()
ax[2].plot(val_losses, label="Val loss")
ax[2].legend()
plt.show()


# PDP (structured)

## Create the config

In [ ]:
from pquant import pdp_config
config = pdp_config()
config.training_parameters.epochs = 100
config.training_parameters.pretraining_epochs = 20
config.training_parameters.fine_tuning_epochs = 100
# 8 bit data and weights
# For data default k=0
config.quantization_parameters.default_data_integer_bits = 3.
config.quantization_parameters.default_data_fractional_bits = 5.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 7.
config.quantization_parameters.overflow_mode_data = "SAT"
config.quantization_parameters.overflow_mode_parameters = "SAT"
config.quantization_parameters.granularity = "per_channel"
config.pruning_parameters.structured_pruning = True
config.pruning_parameters.sparsity = 0.8
# Reach target sparsity around epoch 67: 0.015 * 67 > 1
config.pruning_parameters.epsilon = 0.015

## Build the model and put to GPU

In [ ]:
model = build_model(config, prune_output=False)
model(torch.rand(1, 16)) # Call once to build Keras layers
model.to("cuda") # Created Keras layers might not be in CUDA, do this again just in case
torch._dynamo.config.cache_size_limit = 64
compiled_model = torch.compile(model, dynamic=True)

## Train the model

In [ ]:
from pquant import train_model
reset_history()  # start the plots from this run
BATCH_SIZE = 8192
train_loader, val_loader, test_loader = create_dataloaders(BATCH_SIZE)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lr=1e-3, params=compiled_model.parameters())

trained_model = train_model(model = compiled_model,
                            config = config,
                            train_func = training_loop,
                            valid_func = validate_loop,
                            trainloader = train_loader,
                            device="cuda",
                            valloader = val_loader,
                            loss_function = loss_function,
                            optimizer = optimizer,
                            input_shape=(16,),
                            gather_ebops=True
                                )


## Apply compression to weights one more time
The weights and biases during training are in their non-compressed state, only going through compression during the forward pass. Run the function below to compress the weights and biases one more time, and override the previous non-compressed state of weights and biases.

In [ ]:
from pquant import apply_final_compression
apply_final_compression(trained_model)

## Convert the model to FPGA firmware with hls4ml
The hls4ml converter extracts the quantization parameters from the PQLayers during the conversion.

In [ ]:
from hls4ml.utils import config_from_pytorch_model
from hls4ml.converters import convert_from_pytorch_model
model = trained_model._orig_mod # Run hls4ml conversion with the original eager model instead of the compiled model
model.eval()
output_dir = f"example_result_pdp"
os.makedirs(output_dir, exist_ok = True)

io_type = 'io_parallel'
backend = 'vitis'
target = 'xcvu13p-flga2577-2-e'
model.to("cpu")
hls_config = config_from_pytorch_model(
                model,
                input_shape=tuple([16]),
                granularity='name',
                backend=backend,
                transpose_outputs=True
            )
hls_model = convert_from_pytorch_model(
                model,
                io_type=io_type,
                output_dir=output_dir,
                backend=backend,
                hls_config=hls_config,
                part=target,
            )
hls_model.compile()
res_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml_config = np.mean((np.argmax(res_hls, axis=1) == y_test))

model.to("cuda")
p_torch = model(torch.tensor(X_test).to("cuda")).detach().cpu().numpy()
p_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml = np.mean((np.argmax(p_hls, axis=1) == y_test))
test_accuracy_torch = np.mean((np.argmax(p_torch, axis=1) == y_test))

print(f"Accuracies: hls4ml={test_accuracy_hls4ml_config}, keras={test_accuracy_torch}")


## Make some plots from values gathered during training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,3, figsize=(10,10))

ax[0].plot(remaining_weights, label="Remaining weights %")
ax[0].legend()
ax[1].plot(val_accuracies, label="Val accuracy")
ax[1].legend()
ax[2].plot(val_losses, label="Val loss")
ax[2].legend()
plt.show()


# HGQ

## Create the config

In [ ]:
# Let's use DST config and disable its pruning algorithm
config = dst_config()
config.pruning_parameters.enable_pruning = False
config.quantization_parameters.default_data_integer_bits = 5.
config.quantization_parameters.default_data_fractional_bits = 3.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 7.
config.quantization_parameters.overflow_mode_data = "WRAP"
config.quantization_parameters.overflow_mode_parameters = "SAT_SYM"
# Enable HGQ, with a learned bitwidth for every single weight
config.quantization_parameters.use_high_granularity_quantization = True
config.quantization_parameters.granularity = "per_weight"
config.quantization_parameters.hgq_beta = 1e-6
config.training_parameters.epochs = 2000

## Build the model and put to GPU

In [ ]:
model = build_model(config, prune_output=True, in_out_quant_bits=(1,4,11))
model(torch.rand(1, 16)) # Call once to build Keras layers
model.to("cuda") # Created Keras layers might not be in CUDA, do this again just in case
torch._dynamo.config.cache_size_limit = 64
compiled_model = torch.compile(model, dynamic=True)

## Train the model

In [ ]:
from pquant import train_model
reset_history()  # start the plots from this run
BATCH_SIZE = 33200
train_loader, val_loader, test_loader = create_dataloaders(BATCH_SIZE)
loss_function = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(lr=5e-3, params=compiled_model.parameters())

trained_model = train_model(model = compiled_model,
                            config = config,
                            train_func = training_loop,
                            valid_func = validate_loop,
                            trainloader = train_loader,
                            device="cuda",
                            valloader = val_loader,
                            loss_function = loss_function,
                            optimizer = optimizer,
                            input_shape=(16,),
                            gather_ebops=True
                                )


## Apply compression to weights one more time
The weights and biases during training are in their non-compressed state, only going through compression during the forward pass. Run the function below to compress the weights and biases one more time, and override the previous non-compressed state of weights and biases.

In [ ]:
from pquant import apply_final_compression
apply_final_compression(trained_model)

## Convert the model to FPGA firmware with hls4ml
The hls4ml converter extracts the quantization parameters from the PQLayers during the conversion.

In [ ]:
from hls4ml.utils import config_from_pytorch_model
from hls4ml.converters import convert_from_pytorch_model
model = trained_model._orig_mod # Run hls4ml conversion with the original eager model instead of the compiled model
model.eval()
output_dir = f"example_result_hgq"
os.makedirs(output_dir, exist_ok = True)

io_type = 'io_parallel'
backend = 'vitis'
target = 'xcvu13p-flga2577-2-e'
model.to("cpu")
hls_config = config_from_pytorch_model(
                model,
                input_shape=tuple([16]),
                granularity='name',
                backend=backend,
                transpose_outputs=True
            )
hls_model = convert_from_pytorch_model(
                model,
                io_type=io_type,
                output_dir=output_dir,
                backend=backend,
                hls_config=hls_config,
                part=target,
            )
hls_model.compile()
res_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml_config = np.mean((np.argmax(res_hls, axis=1) == y_test))

model.to("cuda")
p_torch = model(torch.tensor(X_test).to("cuda")).detach().cpu().numpy()
p_hls = hls_model.predict(np.ascontiguousarray(X_test))
test_accuracy_hls4ml = np.mean((np.argmax(p_hls, axis=1) == y_test))
test_accuracy_torch = np.mean((np.argmax(p_torch, axis=1) == y_test))

print(f"Accuracies: hls4ml={test_accuracy_hls4ml_config}, keras={test_accuracy_torch}")


## Make some plots from values gathered during training

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1,3, figsize=(20,5))

ax[0].plot(remaining_weights, label="Remaining weights %")
ax[0].legend()
ax[1].plot(val_accuracies, label="Val accuracy")
ax[1].legend()
ax[2].plot(val_losses, label="Val loss")
ax[2].legend()
plt.show()


In [ ]:
get_ebops(model)

In [ ]:
import matplotlib.pyplot as plt

bitwidths = {}
sparsities = {}
for n, m in model.named_modules():
    if isinstance(m, PQDense):
        weight = m.weight.detach()
        k, i, f = (t.detach().cpu().numpy() for t in m.get_weight_quantization_bits())
        b = np.maximum(i + f, 0)
        bitwidths[n] = np.broadcast_to(np.where(b > 0, k + b, 0), weight.shape).flatten()
        sparsities[n] = float((weight == 0).sum() / weight.numel())

fig, axes = plt.subplots(2, 2, figsize=(10, 8))
for ax, (name, bits) in zip(axes.flat, bitwidths.items()):
    ax.hist(bits, bins=np.arange(bits.min(), bits.max() + 2) - 0.5, edgecolor="black")
    ax.set_xticks(np.arange(bits.min(), bits.max() + 1))
    ax.set_title(f"{name}: weight bitwidths")
    ax.set_xlabel("bits")
    ax.set_ylabel("weights")
fig.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(list(sparsities.keys()), [100 * s for s in sparsities.values()])
ax.set_ylabel("zero weights (%)")
ax.set_title("sparsity per layer")
plt.show()


# Layer replacement

In this section we load in a model from the torchvision repository and replace its layers with PQLayers, using our config.

In [ ]:
from pquant import dst_config
import torch

config = dst_config()
config.training_parameters.epochs = 50
config.training_parameters.fine_tuning_epochs = 50
# 8 bit data and weights
# For data default k=0, weights k=1
config.quantization_parameters.default_data_integer_bits = 3.
config.quantization_parameters.default_data_fractional_bits = 5.
config.quantization_parameters.default_weight_integer_bits = 0.
config.quantization_parameters.default_weight_fractional_bits = 7.
config.quantization_parameters.overflow_mode_data = "SAT"
config.quantization_parameters.overflow_mode_parameters = "SAT"

from torchvision.models import resnet18

model = resnet18()
# PQuantML doesn't support AdaptiveAvgPool2d, replace it manually
model.avgpool = torch.nn.AvgPool2d(7)
from torchinfo import summary
summary(model, (1,3,224,224))

Let's add the compression layers and run summary again. In this case we see 4x the parameters, because these parameters now include the super layer's weight (which is not used at all), the PQConv2d layer's _weight, and the mask and the learned threshold tensor of the DST layer, which are weight shaped.


In [ ]:
from pquant import add_compression_layers
add_compression_layers(model, config)
summary(model, (1,3,224,224))

# Hyperparameter optimization

In [ ]:
from pquant import dst_config
from pquant import get_ebops
from pquant.core.hyperparameter_optimization import TuningTask
import torch.nn as nn

def build_model_noncompressed():
  # Training data range between -6.18 and 8.96, set 4 integer bits for model input quantizer above to cover whole range
    class Model(torch.nn.Module):
        def __init__(self):
            super().__init__()
            self.dense1 = nn.Linear(16, 64)
            self.dense2 = nn.Linear(64, 32)
            self.dense3 = nn.Linear(32, 32)
            self.dense4 = nn.Linear(32, 5)

        def forward(self, x):
            x = F.relu(self.dense1(x))
            x = F.relu(self.dense2(x))
            x = F.relu(self.dense3(x))
            x = self.dense4(x)
            return x
    return Model()


config = dst_config()
# Tune the DST pruning parameter alpha, values = [start, end, step_size]
config.hpo_parameters.hyperparameter_search.numerical["alpha"] = [5e-7, 5e-4, 5e-7]
# Tune the number of epochs in the training stage, values = [start, end, step_size]
config.hpo_parameters.hyperparameter_search.numerical["epochs"] = [10, 20, 10]
config.hpo_parameters.experiment_name = "experiment_name"
config.hpo_parameters.num_trials = 2


# Create functions that create optimizer and scheduler. Both are needed but scheduler can return None

def get_optimizer(config, model):
    return torch.optim.Adam(lr=1e-3, params=model.parameters())

def get_scheduler(config, optimizer):
    return None

"""
MLflow specific settings, not using it in the tutorial

tuner.set_enable_mlflow()
#Set tracking uri of MLflow, NGT MLflow or local one for instance
tuner.set_tracking_uri("https://ngt.cern.ch/models")
#Set user details if you use NGT MLflow
tuner.set_user("your_email@cern.ch", "your_access_token")
"""

# Include same keywords as in the validation loop; the objective is measured on the validation set
def calculate_accuracy(model, valloader, **kwargs):
    accuracies = []
    for data in valloader:
            inputs, labels = data
            inputs, labels = inputs.to("cuda"), labels.to("cuda")
            inputs = torch.reshape(inputs, (inputs.shape[0], -1))
            outputs = model(inputs)
            acc = torch.mean((torch.argmax(outputs, dim=1) == torch.argmax(labels, dim=1)).float()).cpu().numpy()
            accuracies.append(acc)
    return np.mean(accuracies)



tuner = TuningTask(config)
tuner.set_storage_db("sqlite:///optuna_study.db")
# Set training function for your model, must be callable
tuner.set_training_function(training_loop)
# Set validation function for your model, must be callable
tuner.set_validation_function(validate_loop)
# Set objective function for the finetuning, must be callable
tuner.set_objective_function(name="accuracy", fn=calculate_accuracy, direction="maximize")
tuner.set_objective_function(name="ebops", fn=get_ebops, direction="minimize")
# Set hyperparameters
tuner.set_hyperparameters()
# Set optimizer function, must be callable
tuner.set_optimizer_function(get_optimizer)
# Set scheduler function, must be callable
tuner.set_scheduler_function(get_scheduler)


model = build_model_noncompressed()
model = model.to("cuda")
loss_function = torch.nn.CrossEntropyLoss()
BATCH_SIZE = 1024
train_loader, val_loader, test_loader = create_dataloaders(BATCH_SIZE)

reset_history()  # statistics of all trials of this optimization
best_trials = tuner.run_optimization(model,
                        trainloader=train_loader,
                        valloader=val_loader,
                        loss_function=loss_function,
                        input_shape=(16,),
                        gather_ebops=True)

In [ ]:
best_trials